# US News Rankings â€” Eight-Model Causal Comparison

Recovers the implicit feature weights US News uses for its business-school ranking. Fits eight OLS models (bootstrap CIs + ElasticNet robustness check) across the 2Ã—2Ã—2 grid of ProfessionSalaryRank in/out Ã— GMAT Final 1 / Final 2 Ã— 2026-only / both years. Outputs per-model artifacts to `outputs/per_model/`, a master comparison to `outputs/summary_comparison.xlsx`, and a written recommendation to `outputs/final_report.md`.

In [1]:
import os, json, pickle
from pathlib import Path
from typing import List, Optional, Tuple, Dict, Any
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, ElasticNetCV
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from scipy.stats import norm, spearmanr, rankdata
from joblib import Parallel, delayed

NEW_MODELS_DIR = Path(r"D:\work\US news\notebooks\new_models")
DATA_FILE = NEW_MODELS_DIR / "US News Data 2025 - 2026.xlsx"
OUTPUT_DIR = NEW_MODELS_DIR / "outputs"
PER_MODEL_DIR = OUTPUT_DIR / "per_model"
PER_MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_BOOTSTRAP = 2000
CV_FOLDS = 5

np.random.seed(RANDOM_SEED)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [2]:
RENAME_MAP = {
    'school_info.school_name': 'School',
    'school_info.us_news_rank': 'Rank',
    'school_info.us_news_overall_score': 'OverallScore',
    'admissions_and_enrollment.acceptance_rate': 'AcceptanceRate',
    'ranking_scores_two_year_averages.avg_starting_salary_and_bonus_two_yr_avg': 'AvgSalaryBonus',
    'ranking_scores_two_year_averages.salaries_by_profession_indicator_rank': 'ProfessionSalaryRank',
    'ranking_scores_two_year_averages.fulltime_employed_3_months_after_two_yr_avg': 'Employed3Mo',
    'ranking_scores_two_year_averages.fulltime_employed_at_graduation_two_yr_avg': 'EmployedAtGrad',
    'ranking_scores_two_year_averages.peer_assessment_score_out_of_5': 'PeerScore',
    'ranking_scores_two_year_averages.recruiter_assessment_score_out_of_5': 'RecruiterScore',
    'ranking_scores_two_year_averages.median_undergraduate_gpa': 'MedianGPA',
    'ranking_scores_two_year_averages.median_gmat_score_fulltime_new': 'GMAT_New',
    'ranking_scores_two_year_averages.median_gmat_score_fulltime_old': 'GMAT_Old',
    'GMAT Final 1': 'GMAT_Final_1',
    'GMAT Final 2': 'GMAT_Final_2',
}

def load_year(year: int) -> pd.DataFrame:
    df = pd.read_excel(DATA_FILE, sheet_name=str(year))
    df = df.rename(columns=RENAME_MAP)
    df['GMAT_Final_2'] = pd.to_numeric(df['GMAT_Final_2'], errors='coerce')
    df['GMAT_Final_1'] = pd.to_numeric(df['GMAT_Final_1'], errors='coerce')
    df['Year'] = year
    return df

df_2026 = load_year(2026)
df_2025 = load_year(2025)
print(f"2026: shape={df_2026.shape}, GMAT_Final_2 NaN={df_2026['GMAT_Final_2'].isna().sum()}")
print(f"2025: shape={df_2025.shape}, GMAT_Final_2 NaN={df_2025['GMAT_Final_2'].isna().sum()}")

2026: shape=(122, 16), GMAT_Final_2 NaN=65
2025: shape=(121, 16), GMAT_Final_2 NaN=61


In [3]:
def knn_impute_missing(df: pd.DataFrame, impute_cols: List[str],
                       reference_cols: List[str], n_neighbors: int = 5) -> pd.DataFrame:
    """Standard-scale, KNN-impute, inverse-scale. Only `impute_cols` are written back."""
    work = list(dict.fromkeys(impute_cols + reference_cols))
    sub = df[work].astype(float).copy()
    scaler = StandardScaler()
    sub_scaled = pd.DataFrame(
        scaler.fit_transform(sub.fillna(sub.median(numeric_only=True))),
        columns=work, index=sub.index,
    )
    for c in impute_cols:
        sub_scaled.loc[sub[c].isna(), c] = np.nan
    imputer = KNNImputer(n_neighbors=n_neighbors, weights='distance')
    imputed_scaled = pd.DataFrame(imputer.fit_transform(sub_scaled), columns=work, index=sub.index)
    imputed = pd.DataFrame(scaler.inverse_transform(imputed_scaled), columns=work, index=sub.index)
    out = df.copy()
    for c in impute_cols:
        out[c] = imputed[c]
    return out

In [4]:
LOG_VARS = ['AvgSalaryBonus', 'GMAT_Combined']
LOGIT_VARS = ['EmployedAtGrad', 'Employed3Mo', 'AcceptanceRate']
INV_NORM_VARS = ['ProfessionSalaryRank']

EXPECTED_SIGN_RAW = {
    'PeerScore': '+', 'RecruiterScore': '+', 'MedianGPA': '+',
    'AvgSalaryBonus': '+', 'GMAT_Combined': '+',
    'EmployedAtGrad': '+', 'Employed3Mo': '+',
    'AcceptanceRate': '-', 'ProfessionSalaryRank': '-',
}
EXPECTED_SIGN_TRANSFORMED = {
    'PeerScore': '+', 'RecruiterScore': '+', 'MedianGPA': '+',
    'AvgSalaryBonus': '+', 'GMAT_Combined': '+',
    'EmployedAtGrad': '+', 'Employed3Mo': '+',
    'AcceptanceRate': '-',
    'ProfessionSalaryRank': '+',
}


def transform_features(df: pd.DataFrame, features: List[str],
                       rank_n: Optional[Dict[str, float]] = None) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """Apply log1p / logit / inverse-normal transforms to the feature columns."""
    out = df[features].copy().astype(float)
    rank_counts = dict(rank_n) if rank_n else {}

    for col in LOG_VARS:
        if col in out.columns:
            out[col] = np.log1p(out[col])

    for col in LOGIT_VARS:
        if col in out.columns:
            p = out[col].clip(1e-3, 1 - 1e-3)
            out[col] = np.log(p / (1 - p))

    for col in INV_NORM_VARS:
        if col in out.columns:
            if col not in rank_counts:
                rank_counts[col] = float(out[col].max())
            N = rank_counts[col]
            pct = ((out[col] - 0.5) / N).clip(1e-3, 1 - 1e-3)
            out[col] = -1.0 * norm.ppf(pct)

    return out, rank_counts


def cap_outliers(df: pd.DataFrame, features: List[str], lo: float = 0.05, hi: float = 0.05) -> pd.DataFrame:
    """5/95 winsorisation, applied to raw values before transformation."""
    out = df.copy()
    for col in features:
        if col in out.columns:
            lo_v, hi_v = out[col].quantile(lo), out[col].quantile(1 - hi)
            out[col] = out[col].clip(lo_v, hi_v)
    return out

In [5]:
ALWAYS_KEEP = ['PeerScore', 'RecruiterScore', 'MedianGPA',
               'AvgSalaryBonus', 'EmployedAtGrad', 'Employed3Mo', 'AcceptanceRate']


def prepare_model_frame(*, years: List[int], gmat_col: str,
                        include_profession_rank: bool) -> Tuple[pd.DataFrame, List[str]]:
    """Build (clean df, feature list). Returns the *raw* (untransformed, uncapped) features."""
    assert gmat_col in ('GMAT_Final_1', 'GMAT_Final_2')

    frames = [load_year(y) for y in years]
    df = pd.concat(frames, axis=0, ignore_index=True)
    df['MedianGPA'] = df['MedianGPA'].fillna(df['MedianGPA'].min())

    feats = list(ALWAYS_KEEP)
    if include_profession_rank:
        feats.append('ProfessionSalaryRank')
    feats.append('GMAT_Combined')
    df['GMAT_Combined'] = df[gmat_col]

    df = df[['School', 'Year', 'Rank', 'OverallScore'] + feats].copy()

    impute_targets = [c for c in feats if df[c].isna().any()]
    if impute_targets:
        reference_cols = [c for c in feats if c not in impute_targets] or feats
        df = knn_impute_missing(df, impute_cols=impute_targets,
                                reference_cols=reference_cols, n_neighbors=5)
    if 'ProfessionSalaryRank' in feats:
        df['ProfessionSalaryRank'] = df['ProfessionSalaryRank'].round().clip(lower=1)
    assert df[feats].isna().sum().sum() == 0
    return df, feats


_df_m3, _feats_m3 = prepare_model_frame(years=[2026], gmat_col='GMAT_Final_2',
                                        include_profession_rank=True)
print(f"M3 smoke test â€” shape: {_df_m3.shape}, features ({len(_feats_m3)}):")
print('  ' + ', '.join(_feats_m3))
print(f"NaN count in features: {_df_m3[_feats_m3].isna().sum().sum()}")

M3 smoke test â€” shape: (122, 13), features (9):
  PeerScore, RecruiterScore, MedianGPA, AvgSalaryBonus, EmployedAtGrad, Employed3Mo, AcceptanceRate, ProfessionSalaryRank, GMAT_Combined
NaN count in features: 0


In [6]:
def build_design_matrix(df: pd.DataFrame, features: List[str]
                        ) -> Tuple[pd.DataFrame, pd.Series, Dict[str, Any]]:
    """Cap outliers -> transform -> standardise. Returns (X_std, y, fit_params).

    `fit_params` captures everything needed to repeat the same transformation on perturbed /
    out-of-sample rows: rank_counts (for ProfessionSalaryRank inv-normal), per-feature outlier
    caps, the scaler's mean/std.
    """
    df_capped = cap_outliers(df, features)
    df_trans, rank_counts = transform_features(df_capped, features)
    scaler = StandardScaler()
    X_std = pd.DataFrame(
        scaler.fit_transform(df_trans), columns=features, index=df.index,
    )
    fit_params = {
        'rank_counts': rank_counts,
        'caps': {c: (df[c].quantile(0.05), df[c].quantile(0.95)) for c in features},
        'scaler_mean': dict(zip(features, scaler.mean_)),
        'scaler_scale': dict(zip(features, scaler.scale_)),
    }
    return X_std, df['OverallScore'].astype(float), fit_params


def transform_like(df_raw: pd.DataFrame, features: List[str],
                   fit_params: Dict[str, Any]) -> pd.DataFrame:
    """Apply the same cap/transform/standardise as the training fit, on new raw rows.
    Used for sensitivity analysis perturbations."""
    out = df_raw[features].copy().astype(float)
    for c, (lo, hi) in fit_params['caps'].items():
        if c in out.columns:
            out[c] = out[c].clip(lo, hi)
    out, _ = transform_features(out, features, rank_n=fit_params['rank_counts'])
    for c in features:
        out[c] = (out[c] - fit_params['scaler_mean'][c]) / fit_params['scaler_scale'][c]
    return out

In [7]:
def _ols_fit(X: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, float]:
    """Return (coefs, intercept) from OLS on a possibly-resampled (X, y)."""
    lr = LinearRegression().fit(X, y)
    return lr.coef_, lr.intercept_


def fit_ols_bootstrap(X: pd.DataFrame, y: pd.Series,
                      n_iter: int = N_BOOTSTRAP, n_jobs: int = -1,
                      seed: int = RANDOM_SEED) -> Dict[str, Any]:
    """OLS on the full data for point estimates; bootstrap resample for CIs."""
    point_coef, point_intercept = _ols_fit(X.values, y.values)

    rng = np.random.RandomState(seed)
    seeds = rng.randint(0, 2**31 - 1, size=n_iter)

    def _one(s):
        rs = np.random.RandomState(int(s))
        idx = rs.choice(len(X), size=len(X), replace=True)
        return _ols_fit(X.values[idx], y.values[idx])

    out = Parallel(n_jobs=n_jobs)(delayed(_one)(s) for s in seeds)
    boot_coefs = np.array([o[0] for o in out])
    boot_inter = np.array([o[1] for o in out])

    return {
        'beta': pd.Series(point_coef, index=X.columns),
        'intercept': float(point_intercept),
        'boot_coefs': pd.DataFrame(boot_coefs, columns=X.columns),
        'boot_intercept': boot_inter,
    }


def summarise_ols(ols_result: Dict[str, Any]) -> pd.DataFrame:
    boot = ols_result['boot_coefs']
    beta = ols_result['beta']
    lo = boot.quantile(0.025)
    hi = boot.quantile(0.975)
    std = boot.std()
    same_sign = boot.apply(lambda col: float((np.sign(col) == np.sign(beta[col.name])).mean()))
    report = pd.DataFrame({
        'Beta_OLS': beta,
        'Std_Bootstrap': std,
        'Lower_95_CI': lo,
        'Upper_95_CI': hi,
        'Sign_Stability_Pct': (same_sign * 100).round(1),
    })
    report['Is_Significant'] = ~((report['Lower_95_CI'] <= 0) & (report['Upper_95_CI'] >= 0))
    report['Observed_Sign'] = np.where(report['Beta_OLS'] >= 0, '+', '-')
    report['Expected_Sign'] = report.index.map(EXPECTED_SIGN_TRANSFORMED.get)
    report['Sign_Matches'] = report['Observed_Sign'] == report['Expected_Sign']
    return report.reset_index().rename(columns={'index': 'Feature'})


In [8]:
def _enet_fit(X: np.ndarray, y: np.ndarray, seed: int) -> np.ndarray:
    enet = ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1],
                        cv=5, max_iter=10000, n_jobs=1, random_state=seed)
    enet.fit(X, y)
    return enet.coef_


def fit_enet_bootstrap(X: pd.DataFrame, y: pd.Series,
                       n_iter: int = 500, n_jobs: int = -1,
                       seed: int = RANDOM_SEED) -> Dict[str, Any]:
    """Lower n_iter than OLS (ENet is ~50x slower per fit)."""
    rng = np.random.RandomState(seed)
    seeds = rng.randint(0, 2**31 - 1, size=n_iter)

    def _one(s):
        rs = np.random.RandomState(int(s))
        idx = rs.choice(len(X), size=len(X), replace=True)
        return _enet_fit(X.values[idx], y.values[idx], int(s))

    boot = np.array(Parallel(n_jobs=n_jobs)(delayed(_one)(s) for s in seeds))
    boot_df = pd.DataFrame(boot, columns=X.columns)
    return {
        'beta': boot_df.mean(),
        'boot_coefs': boot_df,
    }


def summarise_enet(enet_result: Dict[str, Any]) -> pd.DataFrame:
    boot = enet_result['boot_coefs']
    beta = enet_result['beta']
    return pd.DataFrame({
        'Feature': beta.index,
        'Beta_ENet': beta.values,
        'Sign': np.where(beta.values >= 0, '+', '-'),
        'Sign_Stability_Pct': (boot.apply(
            lambda c: (np.sign(c) == np.sign(beta[c.name])).mean()) * 100).round(1).values,
    })


In [9]:
def compute_vif(X: pd.DataFrame) -> pd.DataFrame:
    """VIF on the design matrix (already standardised, transformed)."""
    Xc = sm.add_constant(X.values)
    vif = []
    for i, name in enumerate(X.columns):
        vif.append({'Feature': name, 'VIF': float(variance_inflation_factor(Xc, i + 1))})
    return pd.DataFrame(vif)


def condition_number(X: pd.DataFrame) -> float:
    s = np.linalg.svd(X.values, compute_uv=False)
    return float(s.max() / s.min())


In [10]:
def compute_importance(X: pd.DataFrame, y: pd.Series, beta: pd.Series,
                       max_shapley_features: int = 12) -> pd.DataFrame:
    """Three importance measures, plus rank-by-each so disagreement is visible."""
    feats = list(X.columns)
    # 1) |beta| share
    beta_share = beta.abs() / beta.abs().sum() * 100

    # 2) Drop-one delta R^2
    full = LinearRegression().fit(X, y)
    r2_full = r2_score(y, full.predict(X))
    drop_one = {}
    for f in feats:
        Xm = X.drop(columns=[f])
        m = LinearRegression().fit(Xm, y)
        drop_one[f] = r2_full - r2_score(y, m.predict(Xm))

    # 3) Shapley (= dominance analysis): average marginal R^2 over all subset orderings.
    if len(feats) > max_shapley_features:
        shapley = {f: np.nan for f in feats}
    else:
        r2_cache = {(): 0.0}
        for k in range(1, len(feats) + 1):
            for combo in combinations(feats, k):
                Xc = X[list(combo)].values
                r2_cache[tuple(sorted(combo))] = r2_score(y, LinearRegression().fit(Xc, y).predict(Xc))
        from math import comb
        K = len(feats)
        shapley = {}
        for f in feats:
            other = [g for g in feats if g != f]
            total = 0.0
            for k in range(0, K):
                weight = 1.0 / (K * comb(K - 1, k))
                for subset in combinations(other, k):
                    with_f = tuple(sorted(list(subset) + [f]))
                    without_f = tuple(sorted(subset))
                    total += weight * (r2_cache[with_f] - r2_cache[without_f])
            shapley[f] = total
        total_r2 = sum(shapley.values())
        shapley = {f: (v / total_r2 * 100 if total_r2 > 0 else 0.0) for f, v in shapley.items()}

    df = pd.DataFrame({
        'Feature': feats,
        'Beta_Share_Pct': beta_share.values,
        'DropOne_DeltaR2': [drop_one[f] for f in feats],
        'Shapley_R2_Share_Pct': [shapley[f] for f in feats],
    })
    df['Rank_By_Beta'] = df['Beta_Share_Pct'].rank(ascending=False, method='min').astype(int)
    df['Rank_By_DropOne'] = df['DropOne_DeltaR2'].rank(ascending=False, method='min').astype(int)
    df['Rank_By_Shapley'] = df['Shapley_R2_Share_Pct'].rank(ascending=False, method='min').astype(int)
    return df


In [11]:
def cv_metrics(df_raw: pd.DataFrame, features: List[str], n_splits: int = CV_FOLDS,
               seed: int = RANDOM_SEED) -> Dict[str, float]:
    """5-fold CV; if multiple Years present, group by School so the same school can't appear
    in both train and test in a fold."""
    y = df_raw['OverallScore'].values
    if df_raw['Year'].nunique() > 1:
        splitter = GroupKFold(n_splits=n_splits)
        splits = list(splitter.split(df_raw, y, groups=df_raw['School']))
    else:
        splits = list(KFold(n_splits=n_splits, shuffle=True, random_state=seed)
                       .split(df_raw))

    r2_list, sp_list, mae_list, rmse_list = [], [], [], []
    for tr, te in splits:
        df_tr, df_te = df_raw.iloc[tr], df_raw.iloc[te]
        X_tr, y_tr, fp = build_design_matrix(df_tr, features)
        X_te = transform_like(df_te[features], features, fp)
        y_te = df_te['OverallScore'].values
        lr = LinearRegression().fit(X_tr, y_tr)
        p = lr.predict(X_te)
        r2_list.append(r2_score(y_te, p))
        sp_list.append(spearmanr(y_te, p).statistic)
        mae_list.append(mean_absolute_error(y_te, p))
        rmse_list.append(float(np.sqrt(mean_squared_error(y_te, p))))

    return {
        'cv_r2_mean': float(np.mean(r2_list)), 'cv_r2_std': float(np.std(r2_list)),
        'cv_spearman_mean': float(np.mean(sp_list)), 'cv_spearman_std': float(np.std(sp_list)),
        'cv_mae_mean': float(np.mean(mae_list)),
        'cv_rmse_mean': float(np.mean(rmse_list)),
    }

In [12]:
def year_stability(df_raw: pd.DataFrame, features: List[str]) -> Dict[str, Any]:
    """Fit OLS on each year separately, return the standardised-beta delta."""
    if df_raw['Year'].nunique() < 2:
        return {}
    coefs_by_year = {}
    for y_ in sorted(df_raw['Year'].unique()):
        sub = df_raw[df_raw['Year'] == y_]
        X_, y_arr, _ = build_design_matrix(sub, features)
        lr = LinearRegression().fit(X_, y_arr)
        coefs_by_year[int(y_)] = pd.Series(lr.coef_, index=features)
    diffs = (coefs_by_year[max(coefs_by_year)] - coefs_by_year[min(coefs_by_year)]).abs()
    return {
        'per_year_betas': {k: v.to_dict() for k, v in coefs_by_year.items()},
        'max_abs_delta': float(diffs.max()),
        'max_delta_feature': str(diffs.idxmax()),
        'delta_beta_l2': float(np.sqrt((diffs ** 2).sum())),
    }

In [13]:
PERTURBATIONS = [-0.10, -0.05, -0.01, 0.01, 0.05, 0.10]
REVERSE_DIRECTION_FEATURES = {'AcceptanceRate', 'ProfessionSalaryRank'}


def _safe_perturb(values: np.ndarray, feature: str, delta: float) -> np.ndarray:
    new = values * (1.0 + delta)
    if feature in ('AcceptanceRate', 'EmployedAtGrad', 'Employed3Mo'):
        new = np.clip(new, 1e-3, 1 - 1e-3)
    elif feature == 'MedianGPA':
        new = np.clip(new, 0.0, 4.0)
    elif feature == 'GMAT_Combined':
        new = np.clip(new, 200.0, 800.0)
    elif feature == 'ProfessionSalaryRank':
        new = np.clip(np.round(new), 1, None)
    return new


def predict_from_raw(df_raw: pd.DataFrame, features: List[str],
                     fit_params: Dict[str, Any], beta: pd.Series, intercept: float) -> np.ndarray:
    X = transform_like(df_raw, features, fit_params)
    return X.values @ beta.values + intercept


def compute_sensitivity(df_raw: pd.DataFrame, features: List[str],
                        fit_params: Dict[str, Any], beta: pd.Series, intercept: float
                        ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    base_pred = predict_from_raw(df_raw, features, fit_params, beta, intercept)
    base_rank = pd.Series(rankdata(-base_pred, method='min'), index=df_raw.index)

    score_rows, rank_rows, rank_imp_rows = [], [], []
    cols = [f'{int(d*100):+d}%' for d in PERTURBATIONS]

    for f in features:
        s_row, r_row, ri_row = {'Feature': f}, {'Feature': f}, {'Feature': f}
        for d, col in zip(PERTURBATIONS, cols):
            df_pert = df_raw.copy()
            df_pert[f] = _safe_perturb(df_pert[f].values, f, d)
            new_pred = predict_from_raw(df_pert, features, fit_params, beta, intercept)
            new_rank = pd.Series(rankdata(-new_pred, method='min'), index=df_pert.index)
            ds = float(np.median(new_pred - base_pred))
            dr = float((new_rank - base_rank).median())
            s_row[col] = ds
            r_row[col] = dr
            # improvement-direction Î”rank: flip sign for reverse-direction features
            sign_flip = -1.0 if f in REVERSE_DIRECTION_FEATURES else 1.0
            # Î”rank where negative = improvement; multiply by -1 so "+ always means better rank"
            # then if the feature is reverse-direction, the perturbation direction is also flipped
            ri_row[col] = -dr * sign_flip
        score_rows.append(s_row); rank_rows.append(r_row); rank_imp_rows.append(ri_row)
    return (pd.DataFrame(score_rows).set_index('Feature'),
            pd.DataFrame(rank_rows).set_index('Feature'),
            pd.DataFrame(rank_imp_rows).set_index('Feature'))


In [14]:
MODEL_CONFIGS = [
    {'id': 'M1', 'include_profession_rank': True,  'gmat_col': 'GMAT_Final_1', 'years': [2026]},
    {'id': 'M2', 'include_profession_rank': True,  'gmat_col': 'GMAT_Final_1', 'years': [2025, 2026]},
    {'id': 'M3', 'include_profession_rank': True,  'gmat_col': 'GMAT_Final_2', 'years': [2026]},
    {'id': 'M4', 'include_profession_rank': True,  'gmat_col': 'GMAT_Final_2', 'years': [2025, 2026]},
    {'id': 'M5', 'include_profession_rank': False, 'gmat_col': 'GMAT_Final_1', 'years': [2026]},
    {'id': 'M6', 'include_profession_rank': False, 'gmat_col': 'GMAT_Final_1', 'years': [2025, 2026]},
    {'id': 'M7', 'include_profession_rank': False, 'gmat_col': 'GMAT_Final_2', 'years': [2026]},
    {'id': 'M8', 'include_profession_rank': False, 'gmat_col': 'GMAT_Final_2', 'years': [2025, 2026]},
]


def run_one_model(cfg: Dict[str, Any]) -> Dict[str, Any]:
    df_raw, feats = prepare_model_frame(
        years=cfg['years'], gmat_col=cfg['gmat_col'],
        include_profession_rank=cfg['include_profession_rank'])
    X, y, fp = build_design_matrix(df_raw, feats)

    ols = fit_ols_bootstrap(X, y, n_iter=N_BOOTSTRAP, n_jobs=-1)
    enet = fit_enet_bootstrap(X, y, n_iter=500, n_jobs=-1)
    coef_report = summarise_ols(ols)
    enet_report = summarise_enet(enet)
    importance = compute_importance(X, y, ols['beta'])
    vif = compute_vif(X)

    train_pred = X.values @ ols['beta'].values + ols['intercept']
    diagnostics = {
        'config': cfg,
        'n_schools': int(len(df_raw)),
        'n_features': len(feats),
        'features': feats,
        'r2_train': float(r2_score(y, train_pred)),
        'adj_r2_train': float(1 - (1 - r2_score(y, train_pred))
                              * (len(y) - 1) / (len(y) - len(feats) - 1)),
        'spearman_train': float(spearmanr(y, train_pred).statistic),
        'mae_train': float(mean_absolute_error(y, train_pred)),
        'rmse_train': float(np.sqrt(mean_squared_error(y, train_pred))),
        'intercept': float(ols['intercept']),
        'condition_number': condition_number(X),
        'max_vif': float(vif['VIF'].max()),
        'bootstrap_iterations': N_BOOTSTRAP,
        **cv_metrics(df_raw, feats),
        **({'year_stability': year_stability(df_raw, feats)} if len(cfg['years']) > 1 else {}),
    }

    sens_score, sens_rank, sens_rank_improve = compute_sensitivity(df_raw, feats, fp,
                                                                   ols['beta'], ols['intercept'])

    out_dir = PER_MODEL_DIR / cfg['id']
    out_dir.mkdir(parents=True, exist_ok=True)
    coef_report.to_csv(out_dir / 'coefficients_ols.csv', index=False)
    enet_report.to_csv(out_dir / 'coefficients_enet.csv', index=False)
    importance.to_csv(out_dir / 'importance.csv', index=False)
    vif.to_csv(out_dir / 'vif.csv', index=False)
    sens_score.to_csv(out_dir / 'sensitivity_score.csv')
    sens_rank.to_csv(out_dir / 'sensitivity_rank.csv')
    sens_rank_improve.to_csv(out_dir / 'sensitivity_rank_improve.csv')
    with open(out_dir / 'diagnostics.json', 'w') as fp_:
        json.dump(diagnostics, fp_, indent=2, default=float)
    with open(out_dir / 'model_ols.pkl', 'wb') as fp_:
        pickle.dump({'beta': ols['beta'], 'intercept': ols['intercept'],
                     'fit_params': fp, 'features': feats}, fp_)
    df_raw.to_parquet(out_dir / 'training_frame.parquet')

    return {
        'config': cfg, 'features': feats, 'df_raw': df_raw,
        'coef_report': coef_report, 'enet_report': enet_report,
        'importance': importance, 'vif': vif,
        'sens_score': sens_score, 'sens_rank': sens_rank, 'sens_rank_improve': sens_rank_improve,
        'diagnostics': diagnostics,
        'ols': ols, 'enet': enet,
    }


In [15]:
def compute_quality_score(res: Dict[str, Any]) -> Dict[str, float]:
    coef = res['coef_report'].set_index('Feature')
    # 1) coef_stability: 1 - mean CV of |beta| (clipped to [0, 1]; CV = std / |mean|)
    cv = (coef['Std_Bootstrap'] / coef['Beta_OLS'].abs().replace(0, np.nan)).fillna(2.0)
    coef_stability = float(np.clip(1 - cv.mean(), 0.0, 1.0))
    # 2) sign_coherence: fraction of features whose sign matches expected
    sign_coherence = float(coef['Sign_Matches'].mean())
    # 3) vif_clean: 1 - min(1, maxVIF/10)
    vif_clean = float(1 - min(1.0, res['diagnostics']['max_vif'] / 10.0))
    # 4) importance_agreement: mean Spearman rho across the three rank columns
    imp = res['importance'].set_index('Feature')[
        ['Rank_By_Beta', 'Rank_By_DropOne', 'Rank_By_Shapley']]
    if imp['Rank_By_Shapley'].isna().any():
        imp = imp[['Rank_By_Beta', 'Rank_By_DropOne']]
    rhos = []
    cols = imp.columns.tolist()
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            r = spearmanr(imp[cols[i]], imp[cols[j]]).statistic
            rhos.append(0.0 if np.isnan(r) else r)
    importance_agreement = float(np.mean(rhos)) if rhos else 0.0
    # 5) bootstrap_sign_stability: mean of Sign_Stability_Pct / 100
    bootstrap_sign_stability = float(coef['Sign_Stability_Pct'].mean() / 100.0)

    composite = float(np.mean([coef_stability, sign_coherence, vif_clean,
                               importance_agreement, bootstrap_sign_stability]))
    return {
        'coef_stability': coef_stability,
        'sign_coherence': sign_coherence,
        'vif_clean': vif_clean,
        'importance_agreement': importance_agreement,
        'bootstrap_sign_stability': bootstrap_sign_stability,
        'composite_score': composite,
    }


In [16]:
results: Dict[str, Dict[str, Any]] = {}
quality: Dict[str, Dict[str, float]] = {}
for cfg in MODEL_CONFIGS:
    print(f"\n=== Running {cfg['id']}: profession={cfg['include_profession_rank']}, "
          f"gmat={cfg['gmat_col']}, years={cfg['years']} ===")
    res = run_one_model(cfg)
    results[cfg['id']] = res
    quality[cfg['id']] = compute_quality_score(res)
    print(f"  R²={res['diagnostics']['r2_train']:.4f}  "
          f"Spearman={res['diagnostics']['spearman_train']:.4f}  "
          f"CV-R²={res['diagnostics']['cv_r2_mean']:.4f}  "
          f"maxVIF={res['diagnostics']['max_vif']:.2f}  "
          f"composite={quality[cfg['id']]['composite_score']:.3f}")


=== Running M1: profession=True, gmat=GMAT_Final_1, years=[2026] ===


  R²=0.9728  Spearman=0.9844  CV-R²=0.9619  maxVIF=17.07  composite=0.587

=== Running M2: profession=True, gmat=GMAT_Final_1, years=[2025, 2026] ===


  R²=0.9058  Spearman=0.9462  CV-R²=0.8773  maxVIF=16.28  composite=0.520

=== Running M3: profession=True, gmat=GMAT_Final_2, years=[2026] ===


  R²=0.9728  Spearman=0.9848  CV-R²=0.9608  maxVIF=16.96  composite=0.569

=== Running M4: profession=True, gmat=GMAT_Final_2, years=[2025, 2026] ===


  R²=0.9266  Spearman=0.9607  CV-R²=0.9032  maxVIF=15.81  composite=0.470

=== Running M5: profession=False, gmat=GMAT_Final_1, years=[2026] ===


  R²=0.9726  Spearman=0.9836  CV-R²=0.9623  maxVIF=6.47  composite=0.720

=== Running M6: profession=False, gmat=GMAT_Final_1, years=[2025, 2026] ===


  R²=0.9040  Spearman=0.9447  CV-R²=0.8801  maxVIF=6.78  composite=0.614

=== Running M7: profession=False, gmat=GMAT_Final_2, years=[2026] ===


  R²=0.9720  Spearman=0.9832  CV-R²=0.9600  maxVIF=7.47  composite=0.693

=== Running M8: profession=False, gmat=GMAT_Final_2, years=[2025, 2026] ===


  R²=0.9260  Spearman=0.9598  CV-R²=0.9064  maxVIF=7.27  composite=0.603


In [17]:
def build_summary_workbook(results, quality, out_path):
    overview_rows = []
    for mid, r in results.items():
        d = r['diagnostics']
        q = quality[mid]
        overview_rows.append({
            'Model': mid,
            'ProfessionRank': r['config']['include_profession_rank'],
            'GMAT': r['config']['gmat_col'],
            'Years': '+'.join(str(y) for y in r['config']['years']),
            'N': d['n_schools'],
            'R2': round(d['r2_train'], 4),
            'AdjR2': round(d['adj_r2_train'], 4),
            'Spearman': round(d['spearman_train'], 4),
            'CV_R2_mean': round(d['cv_r2_mean'], 4),
            'CV_Spearman_mean': round(d['cv_spearman_mean'], 4),
            'MAE': round(d['mae_train'], 3),
            'RMSE': round(d['rmse_train'], 3),
            'MaxVIF': round(d['max_vif'], 2),
            'CondNum': round(d['condition_number'], 2),
            'Composite_Score': round(q['composite_score'], 4),
        })
    overview = pd.DataFrame(overview_rows).set_index('Model').sort_values(
        'Composite_Score', ascending=False)

    scorecard = pd.DataFrame({mid: q for mid, q in quality.items()}).T.round(4)
    scorecard.index.name = 'Model'

    def _wide(field, src='coef_report', from_index='Feature'):
        pieces = []
        for mid, r in results.items():
            sub = r[src].set_index(from_index)[[field]].rename(columns={field: mid})
            pieces.append(sub)
        return pd.concat(pieces, axis=1)

    coef_wide = _wide('Beta_OLS')
    stab_wide = _wide('Sign_Stability_Pct')
    sig_wide  = _wide('Is_Significant')
    sign_aud  = _wide('Sign_Matches')
    pct_wide  = _wide('Beta_Share_Pct', src='importance')
    shap_wide = _wide('Shapley_R2_Share_Pct', src='importance')
    vif_wide  = _wide('VIF', src='vif')
    enet_wide = _wide('Sign', src='enet_report')

    drivers = []
    feats_union = sorted({f for r in results.values() for f in r['features']})
    for f in feats_union:
        models_with = [m for m, r in results.items() if f in r['features']]
        betas = [results[m]['coef_report'].set_index('Feature').loc[f, 'Beta_OLS']
                 for m in models_with]
        sigs = [results[m]['coef_report'].set_index('Feature').loc[f, 'Is_Significant']
                for m in models_with]
        n_pos = sum(1 for b, s in zip(betas, sigs) if s and b > 0)
        n_neg = sum(1 for b, s in zip(betas, sigs) if s and b < 0)
        n_ns  = sum(1 for s in sigs if not s)
        verdict = (
            'Robust positive driver' if n_pos == len(models_with) else
            'Robust negative driver' if n_neg == len(models_with) else
            'Unstable / sign-flips' if n_pos > 0 and n_neg > 0 else
            'Weak (often non-significant)'
        )
        drivers.append({
            'Feature': f, 'N_Models': len(models_with),
            'N_SigPositive': n_pos, 'N_SigNegative': n_neg, 'N_NonSig': n_ns,
            'Mean_Beta': float(np.mean(betas)),
            'Min_Beta': float(np.min(betas)),
            'Max_Beta': float(np.max(betas)),
            'Verdict': verdict,
        })
    drivers_df = pd.DataFrame(drivers).set_index('Feature')

    pert_cols = [f'{int(d*100):+d}%' for d in PERTURBATIONS]
    def _sens_wide(key):
        out = {p: [] for p in pert_cols}
        for mid, r in results.items():
            for p in pert_cols:
                col = r[key][[p]].rename(columns={p: mid})
                out[p].append(col)
        return {p: pd.concat(out[p], axis=1) for p in pert_cols}

    score_wide = _sens_wide('sens_score')
    rank_wide = _sens_wide('sens_rank')
    rank_imp_wide = _sens_wide('sens_rank_improve')

    with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
        overview.to_excel(writer, sheet_name='overview')
        scorecard.to_excel(writer, sheet_name='quality_scorecard')
        drivers_df.to_excel(writer, sheet_name='cross_model_drivers')
        coef_wide.to_excel(writer, sheet_name='coefficients_wide')
        stab_wide.to_excel(writer, sheet_name='coef_stability_wide')
        sig_wide.to_excel(writer, sheet_name='significance_wide')
        sign_aud.to_excel(writer, sheet_name='sign_audit_wide')
        pct_wide.to_excel(writer, sheet_name='pct_contribution_wide')
        shap_wide.to_excel(writer, sheet_name='shapley_wide')
        vif_wide.to_excel(writer, sheet_name='vif_wide')
        enet_wide.to_excel(writer, sheet_name='enet_sign_check')
        for p in pert_cols:
            tag = p.replace('+', 'p').replace('-', 'm')
            score_wide[p].to_excel(writer, sheet_name=f'sens_score_{tag}')
            rank_wide[p].to_excel(writer, sheet_name=f'sens_rank_{tag}')
            rank_imp_wide[p].to_excel(writer, sheet_name=f'sens_rank_imp_{tag}')

    print(f"Wrote {out_path}")


build_summary_workbook(results, quality, OUTPUT_DIR / 'summary_comparison.xlsx')


Wrote D:\work\US news\notebooks\new_models\outputs\summary_comparison.xlsx


In [18]:
def write_final_report(results, quality, out_path):
    # Identify the recommended model: highest composite score, with ties broken by
    # (1) lower max VIF, (2) higher CV R^2.
    rank_df = pd.DataFrame({
        m: {
            'composite': quality[m]['composite_score'],
            'maxVIF': results[m]['diagnostics']['max_vif'],
            'cv_r2': results[m]['diagnostics']['cv_r2_mean'],
        } for m in results
    }).T
    rank_df = rank_df.sort_values(['composite', 'maxVIF', 'cv_r2'],
                                  ascending=[False, True, False])
    winner = rank_df.index[0]
    runner_up = rank_df.index[1]
    win_res = results[winner]

    # Per-feature cross-model summary
    feats_union = sorted({f for r in results.values() for f in r['features']})
    feature_lines = []
    for f in feats_union:
        models_with = [m for m, r in results.items() if f in r['features']]
        coefs = [(m, results[m]['coef_report'].set_index('Feature').loc[f]) for m in models_with]
        betas = np.array([row['Beta_OLS'] for _, row in coefs])
        sigs = np.array([row['Is_Significant'] for _, row in coefs])
        n = len(models_with)
        n_pos = int(((betas > 0) & sigs).sum())
        n_neg = int(((betas < 0) & sigs).sum())
        n_ns  = int((~sigs).sum())
        mean_b = betas.mean()
        sd_b = betas.std()
        feature_lines.append(
            f"- **{f}** — appears in {n}/8 models. "
            f"Significant positive in {n_pos}; significant negative in {n_neg}; "
            f"non-significant in {n_ns}. Mean β = {mean_b:+.3f} (sd {sd_b:.3f})."
        )

    # Multicollinearity findings
    high_vif = []
    for m, r in results.items():
        v = r['vif'].set_index('Feature')['VIF']
        for f, val in v.items():
            if val > 5:
                high_vif.append(f"  - {m} / {f}: VIF = {val:.2f}")

    # OLS <-> ENet sign disagreements
    disagree = []
    for m, r in results.items():
        ols = r['coef_report'].set_index('Feature')['Beta_OLS']
        enet = r['enet_report'].set_index('Feature')['Beta_ENet']
        for f in ols.index:
            if np.sign(ols[f]) != np.sign(enet[f]):
                disagree.append(f"  - {m} / {f}: OLS β = {ols[f]:+.3f}, ENet β = {enet[f]:+.3f}")

    # Year stability findings (both-years configs)
    year_stab = []
    for m, r in results.items():
        ys = r['diagnostics'].get('year_stability')
        if ys:
            year_stab.append(
                f"  - {m}: max |Δβ| between years = {ys['max_abs_delta']:.3f} "
                f"on `{ys['max_delta_feature']}`; L2 = {ys['delta_beta_l2']:.3f}"
            )

    lines = []
    lines.append(f"# US News 8-Model Ranking — Causal Comparison Report\n")
    lines.append("## Executive summary\n")
    cfg = win_res['config']
    lines.append(
        f"**Recommended model: `{winner}`** — ProfessionSalaryRank "
        f"{'included' if cfg['include_profession_rank'] else 'excluded'}, "
        f"GMAT feature = `{cfg['gmat_col']}`, years = {'+'.join(str(y) for y in cfg['years'])}.\n\n"
        f"It has the highest composite causal-quality score "
        f"({quality[winner]['composite_score']:.3f} vs runner-up `{runner_up}` at "
        f"{quality[runner_up]['composite_score']:.3f}). Predictive metrics: training R² "
        f"= {win_res['diagnostics']['r2_train']:.3f}, Spearman ρ = "
        f"{win_res['diagnostics']['spearman_train']:.3f}, CV-R² = "
        f"{win_res['diagnostics']['cv_r2_mean']:.3f}; max VIF = "
        f"{win_res['diagnostics']['max_vif']:.2f}.\n\n"
        "Top three drivers in this model (by Shapley R² share):\n"
    )
    top3 = win_res['importance'].sort_values('Shapley_R2_Share_Pct', ascending=False).head(3)
    for _, row in top3.iterrows():
        lines.append(f"- **{row['Feature']}** — Shapley share {row['Shapley_R2_Share_Pct']:.1f}%, "
                     f"|β| share {row['Beta_Share_Pct']:.1f}%.")
    lines.append("\n")

    lines.append("## Causal-quality scorecard\n")
    lines.append(pd.DataFrame({m: q for m, q in quality.items()}).T.round(3).to_markdown())
    lines.append("\n## Predictive metrics\n")
    pred_table = pd.DataFrame({m: {
        'R²': results[m]['diagnostics']['r2_train'],
        'Adj R²': results[m]['diagnostics']['adj_r2_train'],
        'Spearman': results[m]['diagnostics']['spearman_train'],
        'CV-R²': results[m]['diagnostics']['cv_r2_mean'],
        'CV-Spearman': results[m]['diagnostics']['cv_spearman_mean'],
        'MAE': results[m]['diagnostics']['mae_train'],
        'RMSE': results[m]['diagnostics']['rmse_train'],
        'Max VIF': results[m]['diagnostics']['max_vif'],
        'Cond #': results[m]['diagnostics']['condition_number'],
    } for m in results}).T.round(3)
    lines.append(pred_table.to_markdown())

    lines.append("\n## Per-feature driver summary across all 8 models\n")
    lines.extend(feature_lines)

    lines.append("\n## Multicollinearity findings (VIF > 5)\n")
    lines.append("\n".join(high_vif) if high_vif else "_No features had VIF > 5 in any model._")

    lines.append("\n\n## OLS ↔ ElasticNet sign disagreements\n")
    lines.append("\n".join(disagree) if disagree
                 else "_OLS and ElasticNet agreed on coefficient signs in every (model, feature) cell._")

    lines.append("\n\n## Year-stability (both-years configs)\n")
    lines.append("\n".join(year_stab) if year_stab else "_n/a — applies only to M2/M4/M6/M8._")

    lines.append("\n\n## Recommendation rationale\n")
    lines.append(
        f"`{winner}` wins because:\n"
        f"- **Coefficient stability**: bootstrap CV averaged across features = "
        f"{1 - quality[winner]['coef_stability']:.3f} (lower is better).\n"
        f"- **Sign coherence**: {int(quality[winner]['sign_coherence'] * len(win_res['features']))} "
        f"of {len(win_res['features'])} features have the expected sign on the transformed "
        f"feature.\n"
        f"- **Multicollinearity**: max VIF = {win_res['diagnostics']['max_vif']:.2f} "
        f"({'comfortable' if win_res['diagnostics']['max_vif'] <= 10 else 'elevated; interpret with care'}).\n"
        f"- **Importance agreement**: rank correlation across `|β|`-share, drop-one ΔR², and "
        f"Shapley share = {quality[winner]['importance_agreement']:.3f}.\n"
        f"- **Bootstrap sign stability**: average % of bootstrap iterations where each coefficient "
        f"kept its mean sign = {quality[winner]['bootstrap_sign_stability']*100:.1f}%.\n"
    )

    lines.append("\n## When to pick a different model\n")
    lines.append(
        "- If **predictive accuracy** matters more than coefficient stability, prefer the model "
        f"with the highest CV-R² (see the predictive-metrics table).\n"
        "- If **only 2026 reality** matters (e.g., decisions about next year's targets), prefer a "
        "2026-only config; the both-years configs blend two years of data.\n"
        "- If `ProfessionSalaryRank` is operationally hard to influence, prefer one of the "
        "`Excluded` configs (M5–M8) so the recommended sensitivity matrix focuses on features "
        "the user can actually move.\n"
    )

    lines.append("\n## Caveats\n")
    lines.append(
        "- KNN imputation was performed once on the full per-config frame, not inside CV folds. "
        "This is acceptable because we are not making out-of-sample predictions; the imputed "
        "values are part of the *training reality* for each config. For honest predictive "
        "evaluation in deployment, imputation must be re-fitted per fold.\n"
        "- Both-years configs treat each school-year as an independent observation. If you would "
        "prefer schools weighted equally across years (one row per school, year-averaged), that "
        "is a different model spec and would shift coefficients.\n"
        "- `GMAT_Final_1` has many rows at the placeholder floor value, which artificially "
        "compresses the GMAT distribution and may understate GMAT's true contribution in M1/M2/M5/M6 "
        "relative to the KNN-imputed `GMAT_Final_2` configs.\n"
    )

    with open(out_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"Wrote {out_path}")


write_final_report(results, quality, OUTPUT_DIR / 'final_report.md')


Wrote D:\work\US news\notebooks\new_models\outputs\final_report.md


In [19]:
def display_model_summary(mid: str, r: Dict[str, Any], q: Dict[str, float]) -> None:
    print("=" * 80)
    print(f"  {mid}: profession={r['config']['include_profession_rank']}, "
          f"gmat={r['config']['gmat_col']}, years={'+'.join(str(y) for y in r['config']['years'])}")
    print(f"  Composite causal-quality score: {q['composite_score']:.3f}")
    print(f"  R²={r['diagnostics']['r2_train']:.4f}  Spearman={r['diagnostics']['spearman_train']:.4f}"
          f"  CV-R²={r['diagnostics']['cv_r2_mean']:.4f}  MaxVIF={r['diagnostics']['max_vif']:.2f}")
    print("=" * 80)
    print("\nCoefficients (OLS, standardised, with bootstrap CI):")
    print(r['coef_report'].to_string(index=False, float_format=lambda x: f'{x:+.3f}'))
    print("\nImportance decomposition:")
    print(r['importance'].to_string(index=False, float_format=lambda x: f'{x:+.3f}'))
    print("\nRank sensitivity (improvement direction; positive = better rank):")
    print(r['sens_rank_improve'].round(2).to_string())
    print()


for mid in sorted(results.keys()):
    display_model_summary(mid, results[mid], quality[mid])


  M1: profession=True, gmat=GMAT_Final_1, years=2026
  Composite causal-quality score: 0.587
  R²=0.9728  Spearman=0.9844  CV-R²=0.9619  MaxVIF=17.07

Coefficients (OLS, standardised, with bootstrap CI):
             Feature  Beta_OLS  Std_Bootstrap  Lower_95_CI  Upper_95_CI  Sign_Stability_Pct  Is_Significant Observed_Sign Expected_Sign  Sign_Matches
           PeerScore    +5.255         +0.959       +3.360       +7.077            +100.000            True             +             +          True
      RecruiterScore    +3.639         +0.446       +2.639       +4.423            +100.000            True             +             +          True
           MedianGPA    +2.851         +0.467       +1.959       +3.775            +100.000            True             +             +          True
      AvgSalaryBonus    +8.397         +1.523       +5.252      +11.339            +100.000            True             +             +          True
      EmployedAtGrad    +1.719         +0.549 

In [20]:
def compute_gwu_analysis(res: Dict[str, Any], school_name: str = 'George Washington University'
                         ) -> Optional[Dict[str, Any]]:
    """Build the four GWU-specific tables for one model.

    Returns None if `school_name` is not in the model's training frame; otherwise a dict with:
      - 'global':                 full model coefficient report (already in res['coef_report']) +
                                  Pct_Contribution column from |beta|-share.
      - 'gwu_contribution':       per-feature contribution = beta * GWU_standardized_value.
      - 'gwu_score_sensitivity':  Delta predicted score for GWU only, per feature x perturbation.
      - 'gwu_rank_sensitivity':   Delta predicted rank for GWU, after perturbing only GWU's feature
                                  while leaving every other school's features unchanged.
      - 'gwu_metadata':           dict with the GWU row index, actual/predicted rank+score, year used.
    """
    df_raw = res['df_raw']
    features = res['features']
    coef_report = res['coef_report'].set_index('Feature')
    importance = res['importance'].set_index('Feature')

    gwu_mask = df_raw['School'] == school_name
    if not gwu_mask.any():
        return None

    # For both-years configs, prefer the most recent year (the 2025 row stays as a competitor).
    sub = df_raw[gwu_mask]
    if 'Year' in df_raw.columns and sub['Year'].nunique() > 1:
        latest_year = int(sub['Year'].max())
        gwu_idx = sub[sub['Year'] == latest_year].index[0]
    else:
        gwu_idx = sub.index[0]

    # Load the saved model state to get beta, intercept, fit_params consistently with the
    # artifacts written to disk (rather than re-deriving from in-memory ols dict).
    with open(PER_MODEL_DIR / res['config']['id'] / 'model_ols.pkl', 'rb') as fp_:
        model_state = pickle.load(fp_)
    beta = model_state['beta']
    intercept = model_state['intercept']
    fit_params = model_state['fit_params']

    gwu_row = df_raw.loc[[gwu_idx]]

    # ===== Section 1: Global model report =====
    global_table = coef_report.copy()
    global_table['Pct_Contribution'] = importance['Beta_Share_Pct'].round(2)
    global_table = global_table.reset_index()

    # ===== Section 2: GWU feature contributions =====
    gwu_X = transform_like(gwu_row[features], features, fit_params)
    gwu_std_vals = gwu_X[features].values[0]
    contributions = beta.reindex(features).values * gwu_std_vals
    total_abs = np.abs(contributions).sum() if np.abs(contributions).sum() > 0 else 1.0
    gwu_contribution = pd.DataFrame({
        'Feature': features,
        'GWU_Raw_Value': [float(gwu_row[f].values[0]) for f in features],
        'GWU_Standardized': gwu_std_vals,
        'Beta_OLS': [float(beta[f]) for f in features],
        'GWU_Contribution': contributions,
        'Pct_Contribution_GWU': np.abs(contributions) / total_abs * 100,
        'Is_Significant': [bool(coef_report.loc[f, 'Is_Significant']) for f in features],
        'Direction': ['+' if v >= 0 else '-' for v in contributions],
    })
    gwu_contribution['GWU_Standardized'] = gwu_contribution['GWU_Standardized'].round(3)
    gwu_contribution['Beta_OLS'] = gwu_contribution['Beta_OLS'].round(3)
    gwu_contribution['GWU_Contribution'] = gwu_contribution['GWU_Contribution'].round(3)
    gwu_contribution['Pct_Contribution_GWU'] = gwu_contribution['Pct_Contribution_GWU'].round(2)

    # ===== Section 3: GWU-only score sensitivity =====
    base_gwu_score = float(predict_from_raw(gwu_row, features, fit_params, beta, intercept)[0])

    score_cols = [f'{int(d*100):+d}%' for d in PERTURBATIONS]
    score_rows = []
    for f in features:
        row = {'Feature': f}
        for d, col in zip(PERTURBATIONS, score_cols):
            gwu_pert = gwu_row.copy()
            gwu_pert[f] = _safe_perturb(gwu_pert[f].values, f, d)
            new_score = float(predict_from_raw(gwu_pert, features, fit_params, beta, intercept)[0])
            row[col] = round(new_score - base_gwu_score, 4)
        score_rows.append(row)
    score_sens = pd.DataFrame(score_rows)

    # ===== Section 4: GWU-only rank sensitivity =====
    # Perturb only GWU's feature; all other rows unchanged. Recompute predictions for all rows
    # (only GWU's prediction changes); rank everyone; report Delta rank for GWU.
    base_all_preds = predict_from_raw(df_raw, features, fit_params, beta, intercept)
    base_ranks = pd.Series(rankdata(-base_all_preds, method='min'), index=df_raw.index)
    base_gwu_rank = int(base_ranks.loc[gwu_idx])

    # Cast feature columns to float up-front so single-cell assignment with a
    # perturbed (possibly fractional) value cannot fail on int dtypes.
    df_raw_float = df_raw.copy()
    for f in features:
        df_raw_float[f] = df_raw_float[f].astype(float)

    rank_rows = []
    for f in features:
        row = {'Feature': f}
        for d, col in zip(PERTURBATIONS, score_cols):
            df_pert = df_raw_float.copy()
            current_val = np.array([df_raw_float.at[gwu_idx, f]], dtype=float)
            df_pert.at[gwu_idx, f] = float(_safe_perturb(current_val, f, d)[0])
            new_preds = predict_from_raw(df_pert, features, fit_params, beta, intercept)
            new_ranks = pd.Series(rankdata(-new_preds, method='min'), index=df_pert.index)
            new_gwu_rank = int(new_ranks.loc[gwu_idx])
            row[col] = new_gwu_rank - base_gwu_rank
        rank_rows.append(row)
    rank_sens = pd.DataFrame(rank_rows)

    return {
        'global': global_table,
        'gwu_contribution': gwu_contribution,
        'gwu_score_sensitivity': score_sens,
        'gwu_rank_sensitivity': rank_sens,
        'gwu_metadata': {
            'school': school_name,
            'gwu_idx': int(gwu_idx),
            'year_used': int(df_raw.at[gwu_idx, 'Year']) if 'Year' in df_raw.columns else None,
            'actual_rank': int(df_raw.at[gwu_idx, 'Rank']),
            'actual_score': float(df_raw.at[gwu_idx, 'OverallScore']),
            'predicted_rank': base_gwu_rank,
            'predicted_score': round(base_gwu_score, 3),
        }
    }


def write_gwu_workbook(results: Dict[str, Dict[str, Any]],
                       out_path: Path,
                       school_name: str = 'George Washington University') -> None:
    """Write a per-model workbook: one sheet per M1..M8, each with 4 sections."""
    with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
        for mid in sorted(results.keys()):
            analysis = compute_gwu_analysis(results[mid], school_name=school_name)
            if analysis is None:
                # Write a placeholder sheet noting the school wasn't found.
                pd.DataFrame([{'Note': f'{school_name} not found in this configuration'}]).to_excel(
                    writer, sheet_name=mid, index=False)
                continue
            md = analysis['gwu_metadata']
            cfg = results[mid]['config']

            # Top-of-sheet metadata block
            header_df = pd.DataFrame([
                {'Item': 'Model ID', 'Value': mid},
                {'Item': 'ProfessionSalaryRank included', 'Value': cfg['include_profession_rank']},
                {'Item': 'GMAT feature', 'Value': cfg['gmat_col']},
                {'Item': 'Years used', 'Value': '+'.join(str(y) for y in cfg['years'])},
                {'Item': 'School', 'Value': md['school']},
                {'Item': 'GWU row year', 'Value': md['year_used']},
                {'Item': 'GWU actual rank', 'Value': md['actual_rank']},
                {'Item': 'GWU predicted rank', 'Value': md['predicted_rank']},
                {'Item': 'GWU actual score', 'Value': md['actual_score']},
                {'Item': 'GWU predicted score', 'Value': md['predicted_score']},
            ])

            startrow = 0
            header_df.to_excel(writer, sheet_name=mid, index=False, startrow=startrow)
            startrow += len(header_df) + 2

            sections = [
                ('1. Global model: coefficient, significance, direction, % contribution',
                 analysis['global']),
                ('2. GWU feature contributions (beta x standardized GWU value)',
                 analysis['gwu_contribution']),
                ('3. GWU score sensitivity (delta predicted score for GWU)',
                 analysis['gwu_score_sensitivity']),
                ('4. GWU rank sensitivity (delta rank for GWU; negative = moved up)',
                 analysis['gwu_rank_sensitivity']),
            ]
            for label, table in sections:
                pd.DataFrame([{'Section': label}]).to_excel(
                    writer, sheet_name=mid, index=False, header=False, startrow=startrow)
                startrow += 1
                table.to_excel(writer, sheet_name=mid, index=False, startrow=startrow)
                startrow += len(table) + 3
    print(f"Wrote {out_path}")


In [21]:
write_gwu_workbook(results, OUTPUT_DIR / 'gwu_sensitivity_per_model.xlsx')

Wrote D:\work\US news\notebooks\new_models\outputs\gwu_sensitivity_per_model.xlsx
